In [23]:
import numpy as np
import pandas as pd

# TensorFlow (Keras via TensorFlow ONLY)
import tensorflow as tf
from tensorflow.keras import layers, Model, optimizers, losses, metrics

# Sentence Transformers (works after `pip install tf-keras`)
from sentence_transformers import SentenceTransformer

In [24]:
# movies_df columns:
# movie_id | title | genres | overview | rating | popularity

movies_df = pd.read_csv("movies_features.csv")

In [25]:
# watchlist_df columns:
# user_id | movie_id | progress | timestamp

watchlist_df = pd.read_csv("user_watchlist.csv")

In [26]:
movies_df.head()

,movie_id,title,overview,source_db,genre_action,genre_adventure,genre_animation,genre_comedy,genre_documentary,genre_drama,...,genre_horror,genre_music,genre_mystery,genre_romance,genre_science_fiction,genre_thriller,genre_tv_movie,genre_war,popularity_norm,recency_score
0,39196,Aupa Etxebeste!,The same day that the Etxebeste Family are lea...,european_movies,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0.088407,1.453987e-18
1,54788,Eutsi!,"Two young lovers of cycling, in their thirties...",european_movies,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0.025484,2.809019e-17
2,80187,Gartxot,Gartxot is a Twelfth Century minstrel from the...,european_movies,0,1,1,0,0,1,...,0,0,0,0,0,0,0,0,0.025484,1.407404e-14
3,80550,Bi anai,"At the death of his father, Paulo, the younges...",european_movies,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0.058371,2.780123e-13
4,97056,Stars to Wish Upon,Victoria is a republican widow who enters in t...,european_movies,0,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0.038843,4.371386e-14


In [18]:
user_id = 2

In [8]:
watchlist_df.head()

,id,movie_id,user_id,title,original_title,poster_path,movie_type,watched,source_db
0,12,655187,2.0,Coven,NaN,/lyB3yoQbGO3LiCVuSOD2g17Hr6p.jpg,Streamable,0,european_movies
1,13,80550,2.0,Bi anai,NaN,/sPUU8N0F0wCNapwMMeiWXONFpEz.jpg,Streamable,0,european_movies
2,15,477033,5.0,Errementari: The Blacksmith and the Devil,NaN,/ltpi1uLkvx2BKHWwbpMjqdAtdHn.jpg,Streamable,0,european_movies
3,16,762329,9.0,Ospel,NaN,/oHSOSIt4ZUGTfz8tvR9aPlAI7tt.jpg,Streamable,0,european_movies
4,19,744993,8.0,Nora,NaN,/rz9OvEmsHNPTeyGauqdD5Jhhm5m.jpg,Streamable,0,european_movies


In [27]:
text_model = SentenceTransformer("all-MiniLM-L6-v2")

movies_df["overview"] = movies_df["overview"].fillna("")
overview_embeddings = text_model.encode(
    movies_df["overview"].tolist(),
    show_progress_bar=True
)


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

In [10]:
genre_cols = [c for c in movies_df.columns if c.startswith("genre_")]
genre_vectors = movies_df[genre_cols].values

In [11]:
movie_features = np.hstack([
    overview_embeddings,
    genre_vectors,
    movies_df[["popularity_norm", "recency_score"]].values
])

In [12]:
MOVIE_DIM = movie_features.shape[1]

movie_input = layers.Input(shape=(MOVIE_DIM,))
x = layers.Dense(256, activation="relu")(movie_input)
x = layers.Dense(128, activation="relu")(x)
movie_embedding = layers.Dense(64, activation="relu")(x)

movie_encoder = Model(movie_input, movie_embedding)


2025-12-19 17:11:48.447207: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [13]:
def build_user_embedding(user_id):
    watched_movies = watchlist_df[
        (watchlist_df.user_id == user_id) & (watchlist_df.watched == 1)
    ]["movie_id"]

    indices = movies_df[movies_df.movie_id.isin(watched_movies)].index
    if len(indices) == 0:
        return None

    return movie_encoder.predict(movie_features[indices]).mean(axis=0)


In [15]:
continue_watching = watchlist_df[
    (watchlist_df.user_id == 2) &
    (watchlist_df.watched == 0)
]

In [16]:
def more_like_this(movie_id, top_k=10):
    idx = movies_df[movies_df.movie_id == movie_id].index[0]
    target = movie_encoder.predict(movie_features[idx:idx+1])

    all_embeddings = movie_encoder.predict(movie_features)
    scores = np.dot(all_embeddings, target.T).flatten()

    top_idx = scores.argsort()[::-1][1:top_k+1]
    return movies_df.iloc[top_idx]


In [19]:
user_emb = build_user_embedding(user_id)

scores = np.dot(
    movie_encoder.predict(movie_features),
    user_emb
)

recommended = movies_df.iloc[scores.argsort()[::-1][:20]]


11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 


TypeError: unsupported operand type(s) for *: 'float' and 'NoneType'

In [21]:
user_genres = genre_vectors[indices].mean(axis=0)

genre_scores = np.dot(genre_vectors, user_genres)
top_genre_movies = movies_df.iloc[
    genre_scores.argsort()[::-1][:20]
]

NameError: name 'indices' is not defined